In [21]:
%load_ext autoreload
%autoreload 2
# %flow mode reactive

import os
import pathlib
from colorsys import hls_to_rgb, rgb_to_hls
from contextlib import contextmanager
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objs as go
import seaborn as sns
from numpy.lib.stride_tricks import as_strided

import aeon
import datajoint as dj
from aeon.schema.schemas import social02
from aeon.dj_pipeline.analysis.block_analysis import *
from aeon.dj_pipeline import streams
from aeon.analysis.block_plotting import (
    subject_colors,
    patch_markers_linestyles,
    patch_markers,
    gen_hex_grad,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
streams.UndergroundFeederDeliverPellet()

experiment_name e.g exp0-aeon3,device_serial_number,underground_feeder_install_time time of the underground_feeder placed and started operation at this position,chunk_start datetime of the start of a given acquisition chunk,sample_count number of data points acquired from this stream for a given chunk,timestamps (datetime) timestamps of DeliverPellet data,event
social0.1-aeon3,COM16,2023-11-22 10:19:33,2023-11-22 11:00:00,0,=BLOB=,=BLOB=
social0.1-aeon3,COM16,2023-11-22 10:19:33,2023-11-22 12:00:00,0,=BLOB=,=BLOB=
social0.1-aeon3,COM16,2023-11-22 10:19:33,2023-11-22 13:00:00,0,=BLOB=,=BLOB=
social0.1-aeon3,COM16,2023-11-22 10:19:33,2023-11-22 14:00:00,0,=BLOB=,=BLOB=
social0.1-aeon3,COM16,2023-11-22 10:19:33,2023-11-22 15:00:00,0,=BLOB=,=BLOB=


In [ ]:
streams.UndergroundFeeder

In [18]:
"""Set example block key, and print block patch names and subject names"""

key = {"experiment_name": "social0.2-aeon3", "block_start": "2024-02-11 13:36:10"}
patch_names, subject_names = (BlockSubjectAnalysis.Preference & key).fetch("patch_name", "subject_name")
patch_names = np.unique(patch_names)
subject_names = np.unique(subject_names)
print(patch_names, subject_names)

['Patch1' 'Patch2' 'Patch3'] ['BAA-1104045' 'BAA-1104047']


## Block Plots

### 1. Patch stats: mean, next to boxplots of each pellet threshold per patch.

In [46]:
(BlockAnalysis.Patch & key)

experiment_name e.g exp0-aeon3,block_start,"patch_name e.g. Patch1, Patch2",pellet_count,pellet_timestamps,wheel_cumsum_distance_travelled wheel's cumulative distance travelled,wheel_timestamps,patch_threshold,patch_threshold_timestamps,patch_rate,patch_offset
social0.2-aeon3,2024-02-11 13:36:10,Patch1,1,=BLOB=,=BLOB=,=BLOB=,=BLOB=,=BLOB=,0.002,75.0
social0.2-aeon3,2024-02-11 13:36:10,Patch2,117,=BLOB=,=BLOB=,=BLOB=,=BLOB=,=BLOB=,0.01,75.0
social0.2-aeon3,2024-02-11 13:36:10,Patch3,13,=BLOB=,=BLOB=,=BLOB=,=BLOB=,=BLOB=,0.0033,75.0


In [47]:
"""Fetch subject patch info."""

subj_patch_info = (BlockSubjectAnalysis.Patch & key).fetch(format="frame")
patch_info = (BlockAnalysis.Patch & key).fetch("patch_name", "patch_rate", "patch_offset", as_dict=True)
display(subj_patch_info)
display(patch_info)

in_patch_timestamps  \
experiment_name block_start         patch_name subject_name                                                      
social0.2-aeon3 2024-02-11 13:36:10 Patch1     BAA-1104045   [2024-02-11T13:36:10.000000000, 2024-02-11T13:...   
                                               BAA-1104047   [2024-02-11T13:36:10.000000000, 2024-02-11T13:...   
                                    Patch2     BAA-1104045   [2024-02-11T13:36:10.000000000, 2024-02-11T13:...   
                                               BAA-1104047   [2024-02-11T13:36:10.000000000, 2024-02-11T13:...   
                                    Patch3     BAA-1104045   [2024-02-11T13:36:10.000000000, 2024-02-11T13:...   
                                               BAA-1104047   [2024-02-11T13:36:10.000000000, 2024-02-11T13:...   

                                                             in_patch_time  \
experiment_name block_start         patch_name subject_name                  
social0.2-aeon3 2024-02-11 13:36:10 Patch1     BAA-1104045         6.19999   
                                               BAA-1104047        60.59990   
                                    Patch2     BAA-1104045       995.89900   
                                               BAA-1104047       991.99900   
                                    Patch3     BAA-1104045       168.80000   
                                               BAA-1104047       147.40000   

                                                             pellet_count  \
experiment_name block_start         patch_name subject_name                 
social0.2-aeon3 2024-02-11 13:36:10 Patch1     BAA-1104045              0   
                                               BAA-1104047              1   
                                    Patch2     BAA-1104045             50   
                                               BAA-1104047             67   
                                    Patch3     BAA-1104045              4   
                                               BAA-1104047              9   

                                                                                             pellet_timestamps  \
experiment_name block_start         patch_name subject_name                                                      
social0.2-aeon3 2024-02-11 13:36:10 Patch1     BAA-1104045                                                  []   
                                               BAA-1104047                     [2024-02-11T13:59:15.489791870]   
                                    Patch2     BAA-1104045   [2024-02-11T13:40:22.869247913, 2024-02-11T13:...   
                                               BAA-1104047   [2024-02-11T13:36:38.020127773, 2024-02-11T13:...   
                                    Patch3     BAA-1104045   [2024-02-11T14:14:53.117663860, 2024-02-11T14:...   
                                               BAA-1104047   [2024-02-11T13:55:14.065599917, 2024-02-11T13:...   

                                                                                               patch_threshold  \
experiment_name block_start         patch_name subject_name                                                      
social0.2-aeon3 2024-02-11 13:36:10 Patch1     BAA-1104045                                                  []   
                                               BAA-1104047                                  [836.916554785032]   
                                    Patch2     BAA-1104045   [277.86014860127835, 242.2136142790218, 141.15...   
                                               BAA-1104047   [76.06693472876631, 200.13382552132987, 101.20...   
                                    Patch3     BAA-1104045   [592.5324672337056, 592.5324672337056, 161.635...   
                                               BAA-1104047   [376.0708687324503, 108.62361688727586, 108.62...   

                                                                               wheel_cumsum_distance_travelled

[{'patch_name': 'Patch1', 'patch_rate': 0.002, 'patch_offset': 75.0},
 {'patch_name': 'Patch2', 'patch_rate': 0.01, 'patch_offset': 75.0},
 {'patch_name': 'Patch3', 'patch_rate': 0.0033, 'patch_offset': 75.0}]

In [40]:
"""Convert `subj_patch_info` into a form amenable to plotting."""

reset_subj_patch_info = subj_patch_info.reset_index()  # reset to turn MultiIndex into columns
min_subj_patch_info = reset_subj_patch_info[  # select only relevant columns
    ["patch_name", "subject_name", "pellet_timestamps", "patch_threshold"]
]
min_subj_patch_info = min_subj_patch_info.explode(
    ["pellet_timestamps", "patch_threshold"], ignore_index=True
).dropna()

In [43]:
"""Add patch mean values and block-normalized delivery times to pellet info."""


patch_pel_info

pellet_count  \
experiment_name block_start         patch_name                 
social0.2-aeon3 2024-02-11 13:36:10 Patch1                 1   
                                    Patch2               117   
                                    Patch3                13   

                                                                                pellet_timestamps  \
experiment_name block_start         patch_name                                                      
social0.2-aeon3 2024-02-11 13:36:10 Patch1                        [2024-02-11T13:59:15.489791870]   
                                    Patch2      [2024-02-11T13:36:38.020127773, 2024-02-11T13:...   
                                    Patch3      [2024-02-11T13:55:14.065599917, 2024-02-11T13:...   

                                                                  wheel_cumsum_distance_travelled  \
experiment_name block_start         patch_name                                                      
social0.2-aeon3 2024-02-11 13:36:10 Patch1      [-0.0, -0.007670372101785006, -0.0076703721017...   
                                    Patch2      [-0.0, -0.003068148840714713, -0.0, -0.0015340...   
                                    Patch3      [-0.0, -0.0, -0.00153407442035558, -0.0, -0.00...   

                                                                                 wheel_timestamps  \
experiment_name block_start         patch_name                                                      
social0.2-aeon3 2024-02-11 13:36:10 Patch1      [2024-02-11T13:36:10.000000000, 2024-02-11T13:...   
                                    Patch2      [2024-02-11T13:36:10.000000000, 2024-02-11T13:...   
                                    Patch3      [2024-02-11T13:36:10.000000000, 2024-02-11T13:...   

                                                                                  patch_threshold  \
experiment_name block_start         patch_name                                                      
social0.2-aeon3 2024-02-11 13:36:10 Patch1      [244.8721378489292, 244.8721378489292, 836.916...   
                                    Patch2      [208.98901951348023, 76.06693472876631, 200.13...   
                                    Patch3      [376.0708687324503, 108.62361688727586, 592.53...   

                                                                       patch_threshold_timestamps  \
experiment_name block_start         patch_name                                                      
social0.2-aeon3 2024-02-11 13:36:10 Patch1      [2024-02-11T13:36:10.004000186, 2024-02-11T13:...   
                                    Patch2      [2024-02-11T13:36:10.005983829, 2024-02-11T13:...   
                                    Patch3      [2024-02-11T13:36:10.007999897, 2024-02-11T13:...   

                                                patch_rate  patch_offset  
experiment_name block_start         patch_name                            
social0.2-aeon3 2024-02-11 13:36:10 Patch1          0.0020          75.0  
                                    Patch2          0.0100          75.0  
                                    Patch3          0.0033          75.0

In [42]:
min_subj_patch_info

,patch_name,subject_name,pellet_timestamps,patch_threshold
1,Patch1,BAA-1104047,2024-02-11 13:59:15.489791870,836.916555
2,Patch2,BAA-1104045,2024-02-11 13:40:22.869247913,277.860149
3,Patch2,BAA-1104045,2024-02-11 13:47:08.952127934,242.213614
4,Patch2,BAA-1104045,2024-02-11 13:47:31.118400097,141.154829
5,Patch2,BAA-1104045,2024-02-11 13:48:04.986207962,96.713843
...,...,...,...,...
127,Patch3,BAA-1104047,2024-02-11 14:33:23.647488117,835.74665
128,Patch3,BAA-1104047,2024-02-11 14:33:23.659808159,835.74665
129,Patch3,BAA-1104047,2024-02-11 14:48:22.816319942,423.057663
130,Patch3,BAA-1104047,2024-02-11 14:48:57.314047813,451.151502


In [25]:
reset_subj_patch_info.explode(list(("pellet_timestamps", "patch_threshold")))

,patch_name,subject_name,pellet_timestamps,patch_threshold
0,Patch1,BAA-1104045,NaT,NaN
1,Patch1,BAA-1104047,2024-02-11 13:59:15.489791870,836.916555
2,Patch2,BAA-1104045,2024-02-11 13:40:22.869247913,277.860149
2,Patch2,BAA-1104045,2024-02-11 13:47:08.952127934,242.213614
2,Patch2,BAA-1104045,2024-02-11 13:47:31.118400097,141.154829
...,...,...,...,...
5,Patch3,BAA-1104047,2024-02-11 14:33:23.647488117,835.74665
5,Patch3,BAA-1104047,2024-02-11 14:33:23.659808159,835.74665
5,Patch3,BAA-1104047,2024-02-11 14:48:22.816319942,423.057663
5,Patch3,BAA-1104047,2024-02-11 14:48:57.314047813,451.151502


In [ ]:
reset_subj_patch_info = subj_patch_info.reset_index()  # reset to turn MultiIndex into columns
# Create lists as placeholders for columns for new df
time, patch, threshold, subject = [], [], [], []

for _, row in reset_subj_patch_info.iterrows():
    # Append the time, patch, threshold, and subject name to the lists
    time.extend(row["pellet_timestamps"])
    patch.extend(row["patch_name"])
    threshold.extend(row["patch_threshold"])
    subject.extend(row["subject_name"])

# Create a new df with the lists as columns
df = pd.DataFrame({"time": time, "patch": patch, "threshold": threshold, "subject": subject})

In [17]:
reset_subj_patch_info = subj_patch_info.reset_index()  # reset to turn MultiIndex into columns
reset_subj_patch_info

,experiment_name,block_start,patch_name,subject_name,in_patch_timestamps,in_patch_time,pellet_count,pellet_timestamps,patch_threshold,wheel_cumsum_distance_travelled
0,social0.2-aeon3,2024-02-11 13:36:10,Patch1,BAA-1104045,"[2024-02-11T13:36:10.000000000, 2024-02-11T13:...",6.19999,0,[],[],"[-0.0, -0.007670372101785006, -0.0076703721017..."
1,social0.2-aeon3,2024-02-11 13:36:10,Patch1,BAA-1104047,"[2024-02-11T13:36:10.000000000, 2024-02-11T13:...",60.59990,1,[2024-02-11T13:59:15.489791870],[836.916554785032],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,social0.2-aeon3,2024-02-11 13:36:10,Patch2,BAA-1104045,"[2024-02-11T13:36:10.000000000, 2024-02-11T13:...",995.89900,50,"[2024-02-11T13:40:22.869247913, 2024-02-11T13:...","[277.86014860127835, 242.2136142790218, 141.15...","[-0.0, -0.003068148840714713, 0.0, -0.00153407..."
3,social0.2-aeon3,2024-02-11 13:36:10,Patch2,BAA-1104047,"[2024-02-11T13:36:10.000000000, 2024-02-11T13:...",991.99900,67,"[2024-02-11T13:36:38.020127773, 2024-02-11T13:...","[76.06693472876631, 200.13382552132987, 101.20...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,social0.2-aeon3,2024-02-11 13:36:10,Patch3,BAA-1104045,"[2024-02-11T13:36:10.000000000, 2024-02-11T13:...",168.80000,4,"[2024-02-11T14:14:53.117663860, 2024-02-11T14:...","[592.5324672337056, 592.5324672337056, 161.635...","[-0.0, 0.0, -0.00153407442035558, 0.0, -0.0015..."
5,social0.2-aeon3,2024-02-11 13:36:10,Patch3,BAA-1104047,"[2024-02-11T13:36:10.000000000, 2024-02-11T13:...",147.40000,9,"[2024-02-11T13:55:14.065599917, 2024-02-11T13:...","[376.0708687324503, 108.62361688727586, 108.62...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [16]:
subj_patch_info.index

MultiIndex([('social0.2-aeon3', '2024-02-11 13:36:10', 'Patch1', ...),
            ('social0.2-aeon3', '2024-02-11 13:36:10', 'Patch1', ...),
            ('social0.2-aeon3', '2024-02-11 13:36:10', 'Patch2', ...),
            ('social0.2-aeon3', '2024-02-11 13:36:10', 'Patch2', ...),
            ('social0.2-aeon3', '2024-02-11 13:36:10', 'Patch3', ...),
            ('social0.2-aeon3', '2024-02-11 13:36:10', 'Patch3', ...)],
           names=['experiment_name', 'block_start', 'patch_name', 'subject_name'])

In [9]:
BlockSubjectAnalysis.Patch()

experiment_name e.g exp0-aeon3,block_start,"patch_name e.g. Patch1, Patch2",subject_name,in_patch_timestamps timestamps in which a particular subject is spending time at a particular patch,in_patch_time total seconds spent in this patch for this block,pellet_count,pellet_timestamps,patch_threshold patch threshold value at each pellet delivery,wheel_cumsum_distance_travelled wheel's cumulative distance travelled
social0.1-aeon3,2023-12-01 15:54:55,Patch1,CAA-1120746,=BLOB=,0.0,18,=BLOB=,=BLOB=,=BLOB=
social0.1-aeon3,2023-12-01 15:54:55,Patch1,CAA-1120747,=BLOB=,0.0,12,=BLOB=,=BLOB=,=BLOB=
social0.1-aeon3,2023-12-01 15:54:55,Patch2,CAA-1120746,=BLOB=,0.0,0,=BLOB=,=BLOB=,=BLOB=
social0.1-aeon3,2023-12-01 15:54:55,Patch2,CAA-1120747,=BLOB=,0.0,0,=BLOB=,=BLOB=,=BLOB=
social0.1-aeon3,2023-12-01 15:54:55,Patch3,CAA-1120746,=BLOB=,0.0,18,=BLOB=,=BLOB=,=BLOB=
